# Transfer Learning: VED Hybrid Model to OBD2 Diesel Data

This notebook implements a more defensible version of the idea we discussed:

- use the VED HEV/PHEV dataset as the source domain
- use the OBD2 diesel logger dataset as the target domain
- train only on features that are shared across both datasets
- predict fuel rate for both datasets
- predict battery SOC only for the hybrid VED data

The goal is not to force a hybrid-only model to behave as if a diesel engine has hybrid battery behavior. Instead, the goal is to learn a shared representation that can transfer fuel-rate knowledge while keeping SOC as a hybrid-only target.

## Why this notebook is different

Your earlier test notebook loaded a model trained on hybrid-engine data and applied it directly to a pure diesel dataset. That produced unrealistic predictions because:

- the vehicle domains were different
- many required model inputs were missing in the diesel dataset
- SOC is not a meaningful target for a pure diesel engine in the same hybrid sense

This notebook addresses that by redesigning the workflow:

1. Discover a shared feature space between VED and OBD2.
2. Pretrain a neural network on VED using two outputs: fuel rate and SOC.
3. Fine-tune the same network on a combined dataset where:
   - fuel loss is used for both VED and OBD2 rows
   - SOC loss is used only for VED rows
4. Evaluate VED and OBD2 performance separately.

This still does not guarantee good diesel performance. If the overlap between VED and OBD2 is too small, the model may still struggle. But this approach is much more scientifically meaningful than directly reusing the old hybrid-only feature set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import copy
import glob
import json
import math
import os
import random
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

VED_GLOB = '/content/drive/MyDrive/Data/VED-dataset/VED_HEV_PHEV/*.csv'
OBD2_GLOB = '/content/drive/MyDrive/Data/OBD2-datalogger/Fiesta-TD-Ci/*.csv'

OUTPUT_DIR = '/content/drive/MyDrive/Data/VED-dataset/transfer_learning_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PRETRAIN_EPOCHS = 20
FINETUNE_EPOCHS = 15
BATCH_SIZE = 1024
PRETRAIN_LR = 1e-3
FINETUNE_LR = 3e-4
FUEL_LOSS_WEIGHT = 1.0
SOC_LOSS_WEIGHT = 0.4


In [ ]:
def normalize_name(name):
    name = str(name).strip().lower()
    name = name.replace('[', ' ').replace(']', ' ')
    name = name.replace('(', ' ').replace(')', ' ')
    name = name.replace('%', ' percent ')
    name = name.replace('/', ' ')
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return re.sub(r'_+', '_', name).strip('_')

def resolve_alias(columns, aliases):
    normalized = {normalize_name(col): col for col in columns}
    for alias in aliases:
        found = normalized.get(normalize_name(alias))
        if found is not None:
            return found
    return None

def identity(series):
    return pd.to_numeric(series, errors='coerce')

def fahrenheit_to_celsius(series):
    values = pd.to_numeric(series, errors='coerce')
    return (values - 32.0) * (5.0 / 9.0)

def mph_to_kmh(series):
    values = pd.to_numeric(series, errors='coerce')
    return values * 1.60934

def inhg_to_kpa(series):
    values = pd.to_numeric(series, errors='coerce')
    return values * 3.38639

def gal_per_hr_to_l_per_hr(series):
    values = pd.to_numeric(series, errors='coerce')
    return values * 3.78541

SHARED_FEATURE_SPECS = [
    {
        'name': 'vehicle_speed_kmh',
        'ved_aliases': ['Vehicle Speed[km/h]', 'Vehicle speed[km/h]', 'Speed[km/h]'],
        'obd2_aliases': ['Vehicle speed (MPH)', 'Vehicle speed (km/h)', 'Vehicle speed'],
        'ved_transform': identity,
        'obd2_transform': mph_to_kmh,
    },
    {
        'name': 'latitude_deg',
        'ved_aliases': ['Latitude[deg]', 'Latitude (deg)', 'Latitude'],
        'obd2_aliases': ['Latitude (deg)', 'Latitude'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'longitude_deg',
        'ved_aliases': ['Longitude[deg]', 'Longitude (deg)', 'Longitude'],
        'obd2_aliases': ['Longitude (deg)', 'Longitude'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'engine_rpm',
        'ved_aliases': ['Engine RPM[RPM]', 'Engine RPM', 'RPM'],
        'obd2_aliases': ['Engine RPM (RPM)', 'Engine RPM', 'RPM'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
    {
        'name': 'outside_air_temp_c',
        'ved_aliases': ['OAT[DegC]', 'OAT (DegC)', 'Outside Air Temperature[C]', 'Ambient Temperature[C]'],
        'obd2_aliases': ['Ambient air temperature (degF)', 'Ambient air temperature (degC)', 'Ambient air temperature'],
        'ved_transform': identity,
        'obd2_transform': fahrenheit_to_celsius,
    },
    {
        'name': 'coolant_temp_c',
        'ved_aliases': ['Engine Coolant Temp[C]', 'Engine Coolant Temperature[C]', 'Coolant Temperature[C]'],
        'obd2_aliases': ['Engine coolant temperature (°F)', 'Engine coolant temperature (degF)', 'Engine coolant temperature'],
        'ved_transform': identity,
        'obd2_transform': fahrenheit_to_celsius,
    },
    {
        'name': 'manifold_pressure_kpa',
        'ved_aliases': ['Intake Manifold Absolute Pressure[kPa]', 'Manifold Absolute Pressure[kPa]', 'MAP[kPa]'],
        'obd2_aliases': ['Intake manifold absolute pressure (inHg)', 'Intake manifold absolute pressure (kPa)', 'MAP (kPa)'],
        'ved_transform': identity,
        'obd2_transform': inhg_to_kpa,
    },
    {
        'name': 'load_percent',
        'ved_aliases': ['Calculated load value[%]', 'Calculated load value (%)', 'Engine Load[%]', 'Load[%]'],
        'obd2_aliases': ['Calculated load value (%)', 'Calculated load value [%]', 'Engine load (%)'],
        'ved_transform': identity,
        'obd2_transform': identity,
    },
]

VED_FUEL_ALIASES = ['Fuel Rate[L/hr]', 'Fuel Rate [L/hr]', 'Fuel Rate (L/hr)']
VED_SOC_ALIASES = ['HV Battery SOC[%]', 'HV Battery SOC [%]', 'HV Battery SOC (%)']
VED_SORT_ALIASES = {
    'day': ['DayNum', 'Day Num'],
    'time': ['Timestamp(ms)', 'Timestamp [ms]', 'Timestamp (ms)'],
}

OBD2_FUEL_ALIASES = ['Fuel Rate (gal/hr)', 'Fuel Rate [gal/hr]']
OBD2_TIME_ALIASES = ['Time (sec)', 'Time [sec]', 'Time']

def read_obd2_files(obd2_glob):
    files = sorted(glob.glob(obd2_glob))
    if not files:
        raise FileNotFoundError(f'No OBD2 CSV files found for {obd2_glob}')

    frames = []
    for file in files:
        frame = pd.read_csv(file, skiprows=1, encoding='utf-8-sig', skipinitialspace=True, on_bad_lines='skip')
        frame['source_file'] = Path(file).name
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)

def get_ved_files(ved_glob):
    files = sorted(glob.glob(ved_glob))
    if not files:
        raise FileNotFoundError(f'No VED CSV files found for {ved_glob}')
    return files

def resolve_shared_specs(ved_columns, obd2_columns):
    resolved = []
    for spec in SHARED_FEATURE_SPECS:
        ved_col = resolve_alias(ved_columns, spec['ved_aliases'])
        obd2_col = resolve_alias(obd2_columns, spec['obd2_aliases'])
        if ved_col is not None and obd2_col is not None:
            item = dict(spec)
            item['ved_col'] = ved_col
            item['obd2_col'] = obd2_col
            resolved.append(item)
    return resolved

def build_ved_frame(files, resolved_specs, fuel_col, soc_col, sort_cols):
    required_cols = {fuel_col, soc_col}
    required_cols.update(item['ved_col'] for item in resolved_specs)
    required_cols.update(col for col in sort_cols.values() if col is not None)

    frames = []
    for idx, file in enumerate(files, start=1):
        try:
            frame = pd.read_csv(file, usecols=lambda c: c in required_cols)
            if not frame.empty:
                frames.append(frame)
            if idx % 25 == 0:
                print(f'Loaded {idx} VED files...')
        except Exception as exc:
            print(f'Skipping {file} because of {type(exc).__name__}: {exc}')

    if not frames:
        raise ValueError('No VED frames were loaded.')

    df = pd.concat(frames, ignore_index=True)

    engineered = pd.DataFrame()
    for item in resolved_specs:
        engineered[item['name']] = item['ved_transform'](df[item['ved_col']])

    engineered['fuel_rate_lph'] = pd.to_numeric(df[fuel_col], errors='coerce')
    engineered['soc_percent'] = pd.to_numeric(df[soc_col], errors='coerce')
    engineered['dataset_name'] = 'VED'

    if sort_cols['day'] is not None:
        engineered['ved_day'] = pd.to_numeric(df[sort_cols['day']], errors='coerce')
    else:
        engineered['ved_day'] = np.nan

    if sort_cols['time'] is not None:
        engineered['ved_time'] = pd.to_numeric(df[sort_cols['time']], errors='coerce')
    else:
        engineered['ved_time'] = np.arange(len(engineered))

    return engineered

def build_obd2_frame(obd2_df, resolved_specs, fuel_col, time_col):
    engineered = pd.DataFrame()
    for item in resolved_specs:
        engineered[item['name']] = item['obd2_transform'](obd2_df[item['obd2_col']])

    engineered['fuel_rate_lph'] = gal_per_hr_to_l_per_hr(obd2_df[fuel_col])
    engineered['soc_percent'] = np.nan
    engineered['dataset_name'] = 'OBD2'
    engineered['source_file'] = obd2_df['source_file']
    engineered['trip_time_sec'] = pd.to_numeric(obd2_df[time_col], errors='coerce')
    return engineered

def chron_split(df, train_ratio=0.8, val_ratio=0.1):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    train_df = df.iloc[:train_end].reset_index(drop=True)
    val_df = df.iloc[train_end:val_end].reset_index(drop=True)
    test_df = df.iloc[val_end:].reset_index(drop=True)
    return train_df, val_df, test_df

def split_obd2_by_trip(df):
    trips = sorted(df['source_file'].dropna().unique().tolist())
    if len(trips) < 2:
        raise ValueError('Need at least two OBD2 trip files for fine-tune/test splitting.')

    test_trip = trips[-1]
    train_pool = df[df['source_file'] != test_trip].sort_values(['source_file', 'trip_time_sec']).reset_index(drop=True)
    test_df = df[df['source_file'] == test_trip].sort_values(['source_file', 'trip_time_sec']).reset_index(drop=True)

    cut = int(len(train_pool) * 0.8)
    train_df = train_pool.iloc[:cut].reset_index(drop=True)
    val_df = train_pool.iloc[cut:].reset_index(drop=True)
    return train_df, val_df, test_df

class MultiTaskDataset(Dataset):
    def __init__(self, features, targets, masks):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx], self.masks[idx]

def masked_mse_per_target(pred, target, mask):
    diff2 = (pred - target) ** 2
    weighted = diff2 * mask
    denom = mask.sum(dim=0).clamp_min(1.0)
    losses = weighted.sum(dim=0) / denom
    return losses

class TransferFuelSocNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.head = nn.Linear(32, 2)

    def forward(self, x):
        return self.head(self.backbone(x))

def make_xy_mask(df, feature_names):
    X = df[feature_names].to_numpy(dtype=float)
    y = np.column_stack([
        df['fuel_rate_lph'].to_numpy(dtype=float),
        df['soc_percent'].to_numpy(dtype=float),
    ])
    mask = np.column_stack([
        np.isfinite(df['fuel_rate_lph'].to_numpy(dtype=float)).astype(float),
        np.isfinite(df['soc_percent'].to_numpy(dtype=float)).astype(float),
    ])
    return X, y, mask

def evaluate_predictions(y_true, y_pred, mask, prefix):
    names = ['fuel_rate_lph', 'soc_percent']
    for idx, name in enumerate(names):
        valid = mask[:, idx] > 0
        if valid.sum() == 0:
            print(f'{prefix} | {name}: no valid targets')
            continue

        actual = y_true[valid, idx]
        predicted = y_pred[valid, idx]
        mse = mean_squared_error(actual, predicted)
        rmse = math.sqrt(mse)
        mae = mean_absolute_error(actual, predicted)
        r2 = r2_score(actual, predicted)
        print(f'{prefix} | {name}: MSE={mse:.4f} RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}')

def fit_stage(model, train_loader, val_tensors, optimizer, epochs, stage_name):
    best_state = None
    best_val = float('inf')
    history = {'train_total': [], 'val_total': [], 'train_fuel': [], 'train_soc': [], 'val_fuel': [], 'val_soc': []}

    val_x, val_y, val_mask = [tensor.to(device) for tensor in val_tensors]

    for epoch in range(epochs):
        model.train()
        running_total = 0.0
        running_fuel = 0.0
        running_soc = 0.0
        seen = 0

        for batch_x, batch_y, batch_mask in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            optimizer.zero_grad()
            pred = model(batch_x)
            losses = masked_mse_per_target(pred, batch_y, batch_mask)
            total_loss = FUEL_LOSS_WEIGHT * losses[0] + SOC_LOSS_WEIGHT * losses[1]
            total_loss.backward()
            optimizer.step()

            batch_size = len(batch_x)
            running_total += total_loss.item() * batch_size
            running_fuel += losses[0].item() * batch_size
            running_soc += losses[1].item() * batch_size
            seen += batch_size

        model.eval()
        with torch.no_grad():
            val_pred = model(val_x)
            val_losses = masked_mse_per_target(val_pred, val_y, val_mask)
            val_total = FUEL_LOSS_WEIGHT * val_losses[0] + SOC_LOSS_WEIGHT * val_losses[1]

        train_total = running_total / max(seen, 1)
        train_fuel = running_fuel / max(seen, 1)
        train_soc = running_soc / max(seen, 1)

        history['train_total'].append(train_total)
        history['val_total'].append(val_total.item())
        history['train_fuel'].append(train_fuel)
        history['train_soc'].append(train_soc)
        history['val_fuel'].append(val_losses[0].item())
        history['val_soc'].append(val_losses[1].item())

        if val_total.item() < best_val:
            best_val = val_total.item()
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"{stage_name} Epoch {epoch + 1}/{epochs} | "
            f"train_total={train_total:.4f} train_fuel={train_fuel:.4f} train_soc={train_soc:.4f} | "
            f"val_total={val_total.item():.4f} val_fuel={val_losses[0].item():.4f} val_soc={val_losses[1].item():.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return history


## Load datasets and discover the shared feature space

This cell resolves which features really exist in both datasets. That diagnostic is important. If only a very small number of shared features are found, then even transfer learning will remain limited.

In [ ]:
ved_files = get_ved_files(VED_GLOB)
ved_preview = pd.read_csv(ved_files[0], nrows=5)
ved_columns = list(ved_preview.columns)

obd2_raw = read_obd2_files(OBD2_GLOB)
obd2_columns = list(obd2_raw.columns)

resolved_specs = resolve_shared_specs(ved_columns, obd2_columns)
resolved_feature_names = [item['name'] for item in resolved_specs]

ved_fuel_col = resolve_alias(ved_columns, VED_FUEL_ALIASES)
ved_soc_col = resolve_alias(ved_columns, VED_SOC_ALIASES)
ved_sort_cols = {
    'day': resolve_alias(ved_columns, VED_SORT_ALIASES['day']),
    'time': resolve_alias(ved_columns, VED_SORT_ALIASES['time']),
}

obd2_fuel_col = resolve_alias(obd2_columns, OBD2_FUEL_ALIASES)
obd2_time_col = resolve_alias(obd2_columns, OBD2_TIME_ALIASES)

if ved_fuel_col is None or ved_soc_col is None:
    raise ValueError('Could not resolve the required VED target columns for fuel rate and SOC.')

if obd2_fuel_col is None or obd2_time_col is None:
    raise ValueError('Could not resolve the required OBD2 fuel/time columns.')

print('Resolved shared features:')
for item in resolved_specs:
    print(f"  {item['name']}: VED='{item['ved_col']}' | OBD2='{item['obd2_col']}'")

print('\nVED fuel column:', ved_fuel_col)
print('VED SOC column:', ved_soc_col)
print('VED sort columns:', ved_sort_cols)
print('OBD2 fuel column:', obd2_fuel_col)
print('OBD2 time column:', obd2_time_col)

if len(resolved_feature_names) < 2:
    raise ValueError(
        'Fewer than two shared features were found. The notebook should stop here because transfer learning would not be meaningful.'
    )

ved_df = build_ved_frame(ved_files, resolved_specs, ved_fuel_col, ved_soc_col, ved_sort_cols)
obd2_df = build_obd2_frame(obd2_raw, resolved_specs, obd2_fuel_col, obd2_time_col)

ved_df = ved_df.sort_values(['ved_day', 'ved_time']).reset_index(drop=True)
obd2_df = obd2_df.sort_values(['source_file', 'trip_time_sec']).reset_index(drop=True)

print('\nVED rows before cleaning:', len(ved_df))
print('OBD2 rows before cleaning:', len(obd2_df))

ved_df = ved_df.dropna(subset=['fuel_rate_lph', 'soc_percent']).reset_index(drop=True)
obd2_df = obd2_df.dropna(subset=['fuel_rate_lph']).reset_index(drop=True)

print('VED rows after cleaning:', len(ved_df))
print('OBD2 rows after cleaning:', len(obd2_df))

## Prepare splits and preprocess inputs

VED is split chronologically so the later part of the sequence is not leaked into the earlier training segment.

OBD2 is split by trip so the last trip becomes a held-out diesel test set. The earlier trips are used for fine-tuning and validation.

In [ ]:
ved_train_df, ved_val_df, ved_test_df = chron_split(ved_df, train_ratio=0.8, val_ratio=0.1)
obd2_train_df, obd2_val_df, obd2_test_df = split_obd2_by_trip(obd2_df)

print('VED split sizes:', len(ved_train_df), len(ved_val_df), len(ved_test_df))
print('OBD2 split sizes:', len(obd2_train_df), len(obd2_val_df), len(obd2_test_df))
print('OBD2 held-out trip:', obd2_test_df['source_file'].iloc[0])

X_ved_train, y_ved_train, m_ved_train = make_xy_mask(ved_train_df, resolved_feature_names)
X_ved_val, y_ved_val, m_ved_val = make_xy_mask(ved_val_df, resolved_feature_names)
X_ved_test, y_ved_test, m_ved_test = make_xy_mask(ved_test_df, resolved_feature_names)

X_obd2_train, y_obd2_train, m_obd2_train = make_xy_mask(obd2_train_df, resolved_feature_names)
X_obd2_val, y_obd2_val, m_obd2_val = make_xy_mask(obd2_val_df, resolved_feature_names)
X_obd2_test, y_obd2_test, m_obd2_test = make_xy_mask(obd2_test_df, resolved_feature_names)

imputer = SimpleImputer(strategy='median')
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_ved_train_imp = imputer.fit_transform(X_ved_train)
X_ved_val_imp = imputer.transform(X_ved_val)
X_ved_test_imp = imputer.transform(X_ved_test)
X_obd2_train_imp = imputer.transform(X_obd2_train)
X_obd2_val_imp = imputer.transform(X_obd2_val)
X_obd2_test_imp = imputer.transform(X_obd2_test)

X_ved_train_scaled = scaler_X.fit_transform(X_ved_train_imp)
X_ved_val_scaled = scaler_X.transform(X_ved_val_imp)
X_ved_test_scaled = scaler_X.transform(X_ved_test_imp)
X_obd2_train_scaled = scaler_X.transform(X_obd2_train_imp)
X_obd2_val_scaled = scaler_X.transform(X_obd2_val_imp)
X_obd2_test_scaled = scaler_X.transform(X_obd2_test_imp)

y_ved_train_scaled = scaler_y.fit_transform(y_ved_train)
y_ved_val_scaled = scaler_y.transform(y_ved_val)
y_ved_test_scaled = scaler_y.transform(y_ved_test)

y_obd2_train_fill = y_obd2_train.copy()
y_obd2_val_fill = y_obd2_val.copy()
y_obd2_test_fill = y_obd2_test.copy()
soc_train_mean = float(np.nanmean(y_ved_train[:, 1]))
y_obd2_train_fill[:, 1] = soc_train_mean
y_obd2_val_fill[:, 1] = soc_train_mean
y_obd2_test_fill[:, 1] = soc_train_mean

y_obd2_train_scaled = scaler_y.transform(y_obd2_train_fill)
y_obd2_val_scaled = scaler_y.transform(y_obd2_val_fill)
y_obd2_test_scaled = scaler_y.transform(y_obd2_test_fill)

ved_train_loader = DataLoader(MultiTaskDataset(X_ved_train_scaled, y_ved_train_scaled, m_ved_train), batch_size=BATCH_SIZE, shuffle=True)
ved_val_tensors = (
    torch.tensor(X_ved_val_scaled, dtype=torch.float32),
    torch.tensor(y_ved_val_scaled, dtype=torch.float32),
    torch.tensor(m_ved_val, dtype=torch.float32),
)

X_ft_train = np.vstack([X_ved_train_scaled, X_obd2_train_scaled])
y_ft_train = np.vstack([y_ved_train_scaled, y_obd2_train_scaled])
m_ft_train = np.vstack([m_ved_train, m_obd2_train])
X_ft_val = np.vstack([X_ved_val_scaled, X_obd2_val_scaled])
y_ft_val = np.vstack([y_ved_val_scaled, y_obd2_val_scaled])
m_ft_val = np.vstack([m_ved_val, m_obd2_val])

ft_train_loader = DataLoader(MultiTaskDataset(X_ft_train, y_ft_train, m_ft_train), batch_size=BATCH_SIZE, shuffle=True)
ft_val_tensors = (
    torch.tensor(X_ft_val, dtype=torch.float32),
    torch.tensor(y_ft_val, dtype=torch.float32),
    torch.tensor(m_ft_val, dtype=torch.float32),
)


## Stage 1: Pretrain on VED only

This stage teaches the network a hybrid-domain representation from the VED dataset. Both fuel and SOC losses are active here.

In [ ]:
model = TransferFuelSocNet(input_dim=len(resolved_feature_names)).to(device)
pretrain_optimizer = torch.optim.Adam(model.parameters(), lr=PRETRAIN_LR)
pretrain_history = fit_stage(model, ved_train_loader, ved_val_tensors, pretrain_optimizer, PRETRAIN_EPOCHS, 'Pretrain')


## Stage 2: Fine-tune on combined VED and OBD2

This stage keeps the VED knowledge alive while adapting the shared backbone to diesel fuel-rate behavior.

- fuel loss is active for both datasets
- SOC loss is active only for VED rows because the OBD2 diesel rows have no hybrid SOC target

In [ ]:
finetune_optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR)
finetune_history = fit_stage(model, ft_train_loader, ft_val_tensors, finetune_optimizer, FINETUNE_EPOCHS, 'Finetune')


## Evaluate the final model

The key scientific interpretation should come from comparing:

- VED test performance after fine-tuning
- OBD2 test fuel-rate performance
- the number of shared features that were actually available

If OBD2 performance is still poor and the shared feature count is small, that means the main limitation is missing transferable signal, not just model capacity.

In [ ]:
model.eval()

def predict_scaled(x_scaled):
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        pred_scaled = model(x_tensor).cpu().numpy()
    return scaler_y.inverse_transform(pred_scaled)

ved_test_pred = predict_scaled(X_ved_test_scaled)
obd2_test_pred = predict_scaled(X_obd2_test_scaled)

print('Shared feature count used by this notebook:', len(resolved_feature_names))
print('Shared features:', resolved_feature_names)
print()

evaluate_predictions(y_ved_test, ved_test_pred, m_ved_test, prefix='VED test')
print()
evaluate_predictions(y_obd2_test, obd2_test_pred, m_obd2_test, prefix='OBD2 test')

plt.figure(figsize=(10, 4))
plt.plot(pretrain_history['train_total'], label='Pretrain train')
plt.plot(pretrain_history['val_total'], label='Pretrain val')
plt.plot(range(len(pretrain_history['val_total']) - 1, len(pretrain_history['val_total']) + len(finetune_history['val_total']) - 1), finetune_history['train_total'], label='Finetune train')
plt.plot(range(len(pretrain_history['val_total']) - 1, len(pretrain_history['val_total']) + len(finetune_history['val_total']) - 1), finetune_history['val_total'], label='Finetune val')
plt.title('Training history')
plt.xlabel('Epoch index')
plt.ylabel('Weighted loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

obd2_actual_fuel = y_obd2_test[:, 0]
obd2_pred_fuel = obd2_test_pred[:, 0]
plt.figure(figsize=(12, 4))
plt.plot(obd2_actual_fuel, label='Actual OBD2 fuel rate [L/hr]', linewidth=1.2)
plt.plot(obd2_pred_fuel, label='Predicted OBD2 fuel rate [L/hr]', linewidth=1.2)
plt.title('OBD2 held-out trip fuel-rate prediction')
plt.xlabel('Sample index')
plt.ylabel('Fuel rate [L/hr]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

ved_actual_soc = y_ved_test[:, 1]
ved_pred_soc = ved_test_pred[:, 1]
plt.figure(figsize=(12, 4))
plt.plot(ved_actual_soc[:1000], label='Actual VED SOC [%]', linewidth=1.2)
plt.plot(ved_pred_soc[:1000], label='Predicted VED SOC [%]', linewidth=1.2)
plt.title('VED test SOC prediction sample')
plt.xlabel('Sample index')
plt.ylabel('SOC [%]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'ved_obd2_transfer_model.pth'))
joblib.dump(imputer, os.path.join(OUTPUT_DIR, 'shared_feature_imputer.pkl'))
joblib.dump(scaler_X, os.path.join(OUTPUT_DIR, 'shared_feature_scaler_X.pkl'))
joblib.dump(scaler_y, os.path.join(OUTPUT_DIR, 'shared_feature_scaler_y.pkl'))
with open(os.path.join(OUTPUT_DIR, 'shared_feature_names.json'), 'w') as fh:
    json.dump(resolved_feature_names, fh, indent=2)

obd2_predictions = obd2_test_df.copy()
obd2_predictions['predicted_fuel_rate_lph'] = obd2_pred_fuel
obd2_predictions.to_csv(os.path.join(OUTPUT_DIR, 'obd2_test_predictions.csv'), index=False)

print('Saved outputs to:', OUTPUT_DIR)
obd2_predictions.head()